In [13]:
###---->IMPORTS<---###
import scipy as sp
from scipy import stats
import numpy as np
import random
import pandas as pd
from random import shuffle
import matplotlib.pylab as plt
import seaborn as sns
import re
from tqdm.notebook import tqdm

In [1]:
######################################-----------------##########################################################
######################################-MAIN FUNCTIONS--##########################################################
######################################-----------------##########################################################


def read_file_2_list(filename):
    with open(filename) as file:
        listed = [line.rstrip() for line in file if line.rstrip()]
    return listed

################################################################################################################

def replace_geno(text):
    table = {'0/0':'1|1', '0/1':'1|0', '1/0':'0|1', '1/1':'0|0','./0':'1|1', 
             '0/.':'1|1', './1':'0|0', '1/.':'0|0', '1':'0|0', '0':'1|1'}
    return table.get(text, text)

################################################################################################################

def replace_geno_coding(text):
    table = {'0/0':'0','0/1':'1','1/0':'1','1/1':'2','./.':'nan','.':'nan',
            './0':'0','0/.':'0','./1':'2','1/.':'2','1':'2','0':'0'}
    return table.get(text, text)

################################################################################################################

def get_alleles(geno_samples): 
    alleles_sample = []
    for sample in geno_samples:
        sample = sample.split('/')

        if len(sample) == 1:
            alleles_sample.append(sample[0])
            alleles_sample.append(sample[0])
        else:
            alleles_sample.append(sample[0])
            alleles_sample.append(sample[1])
    return alleles_sample

################################################################################################################

def get_alleles_from_index(header, line, pop):
    index_pop = []
    for sample in pop:
        index_sample = header.index(sample)
        index_pop.append(index_sample)
    geno = [line[i].split(':')[0] for i in index_pop]
    all_alleles = get_alleles(geno)
    return all_alleles

############################################################################################################

def random_allele_sampling(all_samples_coverage, ancestral_allele):
    allele_random = []
    for AD in all_samples_coverage:
        if AD=='.':
            allele_random.append('.')
        elif AD=='0:0':
            allele_random.append('.')
        else:
            ADs = [int(x) for x in AD.split(':')]
            ADx = []
            if str(ancestral_allele) == '0':
                for i in range(0,ADs[0]):
                    ADx.append('0')
                for j in range(0,ADs[1]):
                    ADx.append('1')
            else:
                for i in range(0,ADs[0]):
                    ADx.append('1')
                for j in range(0,ADs[1]):
                    ADx.append('0')
            shuffle(ADx)
            allele_random.append(ADx[0])
    return allele_random

################################################################################################################

def get_ancestral_allele(geno_out, geno1, geno2):
    
    out_geno = [int(i) for i in geno_out if i != '.']
    p1_geno = [int(i) for i in geno1 if i != '.']
    p2_geno = [int(i) for i in geno2 if i != '.']
    
    p12_geno = p1_geno+p2_geno
    total_alleles = p1_geno+p2_geno+out_geno

    if len(set(out_geno)) == 0 and len(set(p12_geno)) == 0: #all missing
        ref_allele = 9
        flag = 'allMiss'

    elif len(set(p12_geno)) == 0: #ingroup pops missing
        ref_allele = int(stats.mode(out_geno)[0])
        flag = 'inMiss'

    elif len(set(out_geno)) == 0 and len(set(p12_geno)) > 0: #outgroup missing
        if len(set(p1_geno)) == 0:
            ref_allele = int(stats.mode(p2_geno)[0]) #pop1 missing
            if len(set(p2_geno)) == 1:
                flag = 'in2FoldAnc'
            else:
                flag = 'in2FoldSeg'
        elif len(set(p2_geno)) == 0: #pop2 missing
            ref_allele = int(stats.mode(p1_geno)[0])
            if len(set(p1_geno)) == 1:
                flag = 'in1FoldAnc'
            else:
                flag = 'in1FoldSeg'
        elif len(set(p1_geno)) == 1 and len(set(p2_geno)) == 1:
            ref_allele = p1_geno[0] #pop1 allele sets as reference => conservative
            if  p1_geno[0] == p2_geno[0]:
                flag = 'InvarOutMiss'
            else:
                flag = 'altFix'
        elif len(set(p1_geno)) == 2 and len(set(p2_geno)) == 1:
            ref_allele = p2_geno[0] 
            flag = 'unfoldFix2outMiss'
        elif len(set(p1_geno)) == 1 and len(set(p2_geno)) == 2:
            ref_allele = p1_geno[0]
            flag = 'unfoldFix1outMiss'
        elif len(set(p1_geno)) == 2 and len(set(p2_geno)) == 2:
            ref_allele = int(stats.mode(p12_geno)[0]) ### not random! Will always be 0 in case of 50/50
            flag = 'inFold'

    elif len(set(out_geno)) == 1 and len(set(p12_geno)) == 1: 
        ref_allele = p12_geno[0] #ingroup pops allele as reference => conservative.
        flag = 'allFix'
    elif len(set(out_geno)) == 1 and len(set(p12_geno)) == 2:
        ref_allele = out_geno[0]
        if len(set(p1_geno)) == 0 and len(set(p2_geno)) > 0:
            flag = 'unfolded2Miss1'
        elif len(set(p1_geno)) > 0 and len(set(p2_geno)) == 0:
            flag = 'unfolded1Miss2'
        elif len(set(p1_geno)) == 1 and len(set(p2_geno)) == 1:
            flag = 'unfoldedAltFix'
        elif len(set(p1_geno)) == 2 and len(set(p2_geno)) == 2:
            flag = 'unfoldedILS'
        elif len(set(p1_geno)) == 1 and len(set(p2_geno)) == 2:
            if p1_geno[0] == out_geno[0]:
                flag = 'unfoldedFixAnc1'
            else:
                flag = 'unfoldedFixDer1'
        elif len(set(p1_geno)) == 2 and len(set(p2_geno)) == 1:
            if p2_geno[0] == out_geno[0]:
                flag = 'unfoldedFixAnc2'
            else:
                flag = 'unfoldedFixDer2'
    elif len(set(out_geno)) == 2 and len(set(p12_geno)) == 1:
        ref_allele = p12_geno[0]
        flag = 'InFixAnc'
    elif len(set(out_geno)) == 2 and len(set(p12_geno)) == 2:
        ref_allele = int(stats.mode(total_alleles)[0]) ### not random! Will always be 0 in case of 50/50
        flag = 'allFold'

    else:
        ref_allele = 9
        print('WARNING: Non biallelic locus in the vcf. Replaced with missing value for all samples')
        print(p1_geno, p2_geno, out_geno)
        flag = 'triall'


    return ref_allele, flag

############################################################################################################

def write_gt(vcf_file, p1=None, p2=None, p0=None, m1=0, m2=0, m0=0, polX='REF', low_cov='NO'):
    #get the variables
    if p1 == None:
        pop1 = []
        print('Pop1 is empty')
    else:
        pop1 = read_file_2_list(p1)
        print('Pop1 includes the following individuals:')
        print(*pop1, sep='\t')
    if p2 == None:
        pop2 = []
        print('Pop2 is empty')
    else:
        pop2 = read_file_2_list(p2)
        print('Pop2 includes the following individuals:')
        print(*pop2, sep='\t')
    if p0 == None:
        pop_out = []
        print('Pop_out is empty')
    else:
        pop_out = read_file_2_list(p0)
        print('Pop_out includes the following individuals:')
        print(*pop_out, sep='\t')

    min_ind_pop1 = m1
    print('Minimum required number of individuals in pop1: ',str(m1))
    min_ind_pop2 = m2 
    print('Minimum required number of individuals in pop2: ',str(m2))
    min_ind_pop_out = m0
    print('Minimum required number of individuals in pop_out: ',str(m0))
    how_to_polarize = polX
    print('Polarization strategy is: ', polX)

    #open the output files
    gt = open(vcf_file.replace('.vcf','.'+how_to_polarize+'.gt'), 'w')
    if low_cov == 'YES':
        print('One random read resampling per individual genotype is switched on')
        gt_cov = open(vcf_file.replace('.vcf','.gt.covRandom1'), 'w')
        
    #set the counter
    valid_loci = 0
    inverted_loci = 0
    missing_loci = 0

    #get the total number of lines to parse
    with open(vcf_file, 'r') as handle:
        num_lines = sum(1 for line in handle)
    
    #go through the vcf file without comments
    with open(vcf_file, 'r') as handle:
        for line in tqdm(handle, total=num_lines):
            line = line.rstrip()
            if line.startswith('##') or line == '':
                continue
            elif line.startswith('#CHR'):
                header = line.split('\t')
                inds_ID = header[9:]
                gt.write('scaffold\tposition\teffect\tvartype\tflag\tref\t'+'\t'.join(inds_ID)+'\n')
                if low_cov == 'YES':
                    gt_cov.write('scaffold\tposition\teffect\tvartype\tflag\tref\t'+'\t'.join(inds_ID)+'\n') #fixed to be removed
            else:
                line = line.replace('|','/') # to correct for phased+unphased
                line = line.split('\t')
                all_geno = [line[i].split(':')[0] for i in range(9,len(line))]
                if low_cov == 'YES':
                    all_coverage = [':'.join(line[i].split(':')[2].split(',')[0:2]) for i in range(9,len(line))]
    
    #get all alleles per pop
                alleles_pop1 = get_alleles_from_index(header, line, pop1)
                alleles_pop2 = get_alleles_from_index(header, line, pop2)
                alleles_pop_out = get_alleles_from_index(header, line, pop_out)
                
                alleles_pop1_nomiss = [i for i in alleles_pop1 if i != '.']
                alleles_pop2_nomiss = [i for i in alleles_pop2 if i != '.']
                alleles_pop_out_nomiss = [i for i in alleles_pop_out if i != '.']
                
                if how_to_polarize == 'POP_OUT':
                    anc_all, flag = get_ancestral_allele(alleles_pop_out, alleles_pop1, alleles_pop2)                    
                elif how_to_polarize == 'POP1':
                    no_outgroup = []
                    no_pop2 = []
                    anc_all, flag = get_ancestral_allele(no_outgroup, alleles_pop1, no_pop2)                    
                elif how_to_polarize == 'POP2':
                    no_outgroup = []
                    no_pop1 = []
                    anc_all, flag = get_ancestral_allele(no_outgroup, no_pop1, alleles_pop2)
                elif how_to_polarize == 'POP1POP2':
                    no_outgroup = []
                    anc_all, flag = get_ancestral_allele(no_outgroup, alleles_pop1, alleles_pop2)
                elif how_to_polarize == 'REF': #do not repolarize
                    anc_all = 0
                    flag = 'reference'
                else: #just in case
                    no_outgroup = []
                    anc_all, flag = get_ancestral_allele(no_outgroup, alleles_pop1, alleles_pop2)
          
    #set missing data condition
                if len(alleles_pop_out_nomiss) > (min_ind_pop_out*2)-1 and len(alleles_pop1_nomiss) > (min_ind_pop1*2)-1 and len(alleles_pop2_nomiss) > (min_ind_pop2*2)-1:
    
    #get scaffold, position et al
                    scaffold = line[0]
                    position = int(line[1])
                    vcf_ref_all = line[3]
                    vcf_alt_all = line[4]

    #get the effect and type of variant
                    if 'ANN' in line[7]:
                        snpeff = line[7].split('ANN=')[1].split('/')
                        valid_loci += 1
                        annotation = snpeff[1]
                        effect = snpeff[2]
                        if 'missense' in annotation:
                            vartype = 'missense'
                        elif 'synonymous' in annotation:
                            vartype = 'synonymous'
                        elif 'intergenic' in annotation:
                            vartype = 'intergenic'
                        elif 'intron' in annotation:
                            vartype = 'intron'
                        elif 'downstream' in annotation:
                            vartype = 'downstream'
                        elif 'upstream' in annotation:
                            vartype = 'upstream'
                        else:
                            vartype = 'else'
                        gt.write(scaffold+'\t'+str(position)+'\t'+effect+'\t'+vartype+'\t'+flag+'\t')
                        if low_cov == 'YES':
                            gt_cov.write(scaffold+'\t'+str(position)+'\t'+effect+'\t'+vartype+'\t'+flag+'\t')

    #invert genotype if needed
                        clean_geno = [i.replace('.', './.') if i == '.' else i for i in all_geno] #not really needed as replace_geno_coding function takes care of missing data
                        if anc_all == 0:
                            new_geno = clean_geno
                            gt.write(vcf_ref_all+'\t')
                            if low_cov == 'YES':
                                gt_cov.write(vcf_ref_all+'\t')
                        else:
                            new_geno_pipe = [replace_geno(i) for i in clean_geno]     
                            new_geno = [i.replace('|','/') for i in new_geno_pipe] 
                            gt.write(vcf_alt_all+'\t')
                            if low_cov == 'YES':
                                gt_cov.write(vcf_alt_all+'\t')
                            inverted_loci += 1

                        light_geno = [replace_geno_coding(i) for i in new_geno]
                        gt.write('\t'.join(light_geno)+'\n')
                        
                        if low_cov == 'YES':
                            allele_random = random_allele_sampling(all_coverage, anc_all)
                            gt_cov.write('\t'.join(allele_random)+'\n')
                else:
                    missing_loci += 1
    
    gt.close()
    if low_cov == 'YES':
        gt_cov.close()
    print(valid_loci, 'loci recorded to file, of which', inverted_loci, 'repolarized to ancestral and ', missing_loci, 'not passing missingness filter')                  

################################################################################################################


In [ ]:
###POLARIZE SNPS IN VCF AND WRITE A GT FILE###

###Run this script to polarize and write the .gt file (polarized genotypes reformatted as 0,1,2)
write_gt(f_vcf, p1, p2, p0, m1, m2, m0, polX, low_cov)

###Variables
f_vcf='<filename>.ann.vcf'   ###Required, positional argument: first in list!
###Freebayes or GATK generated vcf file with sample names as in pop1, pop2, pop_out

###Samples are included in the different pop groups only for polarization. 
###All samples in the vcf are converted, even if not included in any pop group.
###Samples can be included in more than a pop group.
###Play with the group assignment and the polarization flag to customize the polarization
p1='<filename>' ###Default = None ###Text file with one sample name per line. Sample names should be the same as in the vcf file"
p2='<filename>' ###Default = None ###Text file with one sample name per line. Sample names should be the same as in the vcf file"
p0='<filename>' ###Default = None ###Text file with one sample name per line. Sample names should be the same as in the vcf file

###For load estimates better set to no missing data (set same number of individuals as in your pop1, pop2, pop_out lists)
m1=int ###Default = 0 ###Minimum number of individuals in this sample group to use the locus
m2=int ###Default = 0 ###Minimum number of individuals in this sample group to use the locus
m0=int ###Default = 0 ###Minimum number of individuals in this sample group to use the locus

###Set how to polarize: 
###major in pop1 = 'POP1', 
###major in pop2 = 'POP2', 
###major in pop1+pop2 = 'POP1POP2', 
###using the outgroup = 'POP_OUT',
###using the reference in the vcf = 'REF'
###Empty pop lists above results in fallbacks.
polX='POP_OUT' ###Default = 'REF' ### Polarize using outgroup

###Switch on additional resampling of one read per genotype based on coverage for ancient or low coverage data. 
##Needs AD field in the genotype FORMAT in the vcf. Options are 'YES' or 'NO'
low_cov='NO' ###Default = 'NO'